# Resilience & Liquidity Analysis
## Phase II: Exploratory Analysis

**Goal**: Load aggregated station-hour data from Rust ETL and perform initial exploratory analysis.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add src to path
import sys
sys.path.insert(0, '../')

from src.shock_registry import BLACK_SWAN_REGISTRY
from src import visualizations

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load Aggregated Data from Parquet

In [ ]:
# Load from Rust ETL output
parquet_path = Path('../rust_etl/output/station_hour_matrix.parquet')

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
    print(f"✅ Loaded {len(df):,} station-hour records")
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nFirst few rows:")
    df.head(10)
else:
    print("❌ Parquet file not found. Run Rust ETL first: cd rust_etl && cargo run --release")

## 2. Black Swan Events for 2023

In [ ]:
print("🌪️  Black Swan Events Registered:\n")
for i, event in enumerate(BLACK_SWAN_REGISTRY, 1):
    print(f"{i}. {event.name}")
    print(f"   Period: {event.start_date.strftime('%Y-%m-%d')} to {event.end_date.strftime('%Y-%m-%d')}")
    print(f"   Magnitude: {event.magnitude:.2f} (0-1 scale)")
    print(f"   Type: {event.event_type}")
    print(f"   {event.description[:100]}...\n")

## 3. Data Quality & Summary Statistics

In [ ]:
print("📊 Data Quality Report:")
print(f"\nMissing Values:")
print(df.isnull().sum())
print(f"\nData Types:")
print(df.dtypes)
print(f"\nBasic Statistics:")
df.describe()

## 4. Temporal Patterns

In [ ]:
# Aggregate by hour across all stations
if 'hour_bucket' in df.columns:
    hourly_agg = df.groupby('hour_bucket').agg({
        'trip_count': 'sum',
        'avg_duration': 'mean',
        'member_ratio': 'mean'
    }).reset_index()
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(range(len(hourly_agg)), hourly_agg['trip_count'], marker='o', linewidth=2)
    ax.set_xlabel('Hour Bucket (chronological)')
    ax.set_ylabel('Total Trips')
    ax.set_title('Daily Trip Volume Pattern (2023)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Peak hour volume: {hourly_agg['trip_count'].max():,.0f} trips")
    print(f"Average hour volume: {hourly_agg['trip_count'].mean():,.0f} trips")
else:
    print("⚠️  No hour_bucket column found in data")

## 5. Station-Level Diversity

In [ ]:
if 'start_station_id' in df.columns:
    n_stations = df['start_station_id'].nunique()
    print(f"Unique Stations: {n_stations}")
    
    # Top 10 busiest stations
    top_stations = df.groupby('start_station_id').agg({
        'trip_count': 'sum',
        'station_name': 'first' if 'station_name' in df.columns else 'nunique'
    }).sort_values('trip_count', ascending=False).head(10)
    
    print(f"\nTop 10 Busiest Stations:")
    print(top_stations)

---

## Next Steps

✅ Phase I (Rust ETL) Complete

🚧 Phase II (Bayesian Inference):
1. **Notebook 02**: Run Orbit BSTS modeling on individual stations
2. **Notebook 03**: Analyze shock sensitivity and impulse responses
3. **Visualizations**: Generate 8 analytical charts
4. **Risk Metrics**: Compute Expected Shortfall and Resilience scores
